In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas  as pd
import pywifi

In [2]:
import time
from datetime import datetime
import openpyxl

def scan_wifi(interval) :
    wifi = pywifi.PyWiFi()
    iface = wifi.interfaces()[0]
    iface.scan()
    time.sleep(interval) # attend une durée “interval” pendant laquelle les réseaux sont observés
    networks = iface.scan_results()
    return networks ; # → networks = scan_wifi(2)

# relevé du temps de mesure
current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-5]

# affichage des réseaux Wi-Fi détectés
# BSSID = adresse MAC du point d’accès

networks = scan_wifi(2)

for network in networks:
    print(f"Time: {current_time}, SSID: {network.ssid}, BSSID: {network.bssid}, RSSI: {network.signal} dBm")
    
data = {"Time":[], "SSID":[], "BSSID":[], "RSSI":[]}
df = pd.DataFrame(data)


for network in networks:
    df.loc[len(df)] = [current_time, network.ssid, network.bssid, network.signal]
    df.to_excel("RSSI0.xlsx",index=False,engine="openpyxl")
    print("C’est fait !")

Time: 2026-02-13 10:11:57.1, SSID: UTTetudiants, BSSID: 84:b2:61:1f:b5:37:, RSSI: -59 dBm
Time: 2026-02-13 10:11:57.1, SSID: UTTetudiants, BSSID: f8:6b:d9:5e:10:c8:, RSSI: -42 dBm
Time: 2026-02-13 10:11:57.1, SSID: UTTetudiants, BSSID: f8:6b:d9:01:93:67:, RSSI: -79 dBm
Time: 2026-02-13 10:11:57.1, SSID: UTTetudiants, BSSID: e4:aa:5d:bf:25:58:, RSSI: -74 dBm
Time: 2026-02-13 10:11:57.1, SSID: UTTetudiants, BSSID: e4:aa:5d:fd:3e:e7:, RSSI: -75 dBm
Time: 2026-02-13 10:11:57.1, SSID: UTTetudiants, BSSID: f8:6b:d9:01:93:68:, RSSI: -83 dBm
Time: 2026-02-13 10:11:57.1, SSID: UTTetudiants, BSSID: f8:6b:d9:01:85:28:, RSSI: -76 dBm
Time: 2026-02-13 10:11:57.1, SSID: UTTetudiants, BSSID: f8:6b:d9:01:85:27:, RSSI: -75 dBm
Time: 2026-02-13 10:11:57.1, SSID: UTTetudiants, BSSID: e4:aa:5d:bf:25:57:, RSSI: -82 dBm
Time: 2026-02-13 10:11:57.1, SSID: UTTetudiants, BSSID: f8:6b:d9:01:8e:e7:, RSSI: -82 dBm
Time: 2026-02-13 10:11:57.1, SSID: UTTetudiants, BSSID: f8:6b:d9:01:8e:e8:, RSSI: -85 dBm
Time: 2026

In [4]:
import time
from datetime import datetime
import pywifi
import json
from pathlib import Path
import tkinter as tk
from tkinter import ttk, messagebox, scrolledtext

class WiFiRSSICollector:
    def __init__(self):
        # Stockage des données : {BSSID: {"ssid": str, "measurements": [(x, y, rssi, time)]}}
        self.data_file = Path("wifi_measurements.json")
        self.data = self.load_data()
        
        # Interface WiFi
        wifi = pywifi.PyWiFi()
        self.iface = wifi.interfaces()[0]
        
    def load_data(self):
        """Charge les données existantes"""
        if self.data_file.exists():
            with open(self.data_file, 'r') as f:
                return json.load(f)
        return {}
    
    def save_data(self):
        """Sauvegarde les données"""
        with open(self.data_file, 'w') as f:
            json.dump(self.data, f, indent=2)
    
    def scan_networks(self, interval=2):
        """Scan les réseaux WiFi"""
        self.iface.scan()
        time.sleep(interval)
        return self.iface.scan_results()
    
    def add_measurement(self, x, y):
        """Ajoute une mesure pour tous les réseaux détectés"""
        networks = self.scan_networks()
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        measurements_added = 0
        
        for network in networks:
            bssid = network.bssid
            ssid = network.ssid if network.ssid else "Hidden"
            rssi = network.signal
            
            # Initialiser le vecteur si nouveau BSSID
            if bssid not in self.data:
                self.data[bssid] = {
                    "ssid": ssid,
                    "measurements": []
                }
            
            # Ajouter la mesure
            self.data[bssid]["measurements"].append({
                "x": x,
                "y": y,
                "rssi": rssi,
                "time": timestamp
            })
            measurements_added += 1
        
        self.save_data()
        return measurements_added, len(networks)
    
    def get_statistics(self):
        """Retourne les statistiques"""
        total_bssids = len(self.data)
        total_measurements = sum(len(info["measurements"]) for info in self.data.values())
        return total_bssids, total_measurements
    
    def export_to_csv(self, filename="wifi_data.csv"):
        """Export simple en CSV sans pandas"""
        with open(filename, 'w', encoding='utf-8') as f:
            f.write("BSSID,SSID,X(m),Y(m),RSSI(dBm),Time\n")
            for bssid, info in self.data.items():
                for m in info["measurements"]:
                    f.write(f"{bssid},{info['ssid']},{m['x']},{m['y']},{m['rssi']},{m['time']}\n")


class WiFiCollectorGUI:
    def __init__(self):
        self.collector = WiFiRSSICollector()
        self.setup_ui()
        
    def setup_ui(self):
        """Création de l'interface"""
        self.root = tk.Tk()
        self.root.title("WiFi RSSI Collector")
        self.root.geometry("700x600")
        
        # Frame principal
        main_frame = ttk.Frame(self.root, padding="10")
        main_frame.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        # === Section de saisie ===
        input_frame = ttk.LabelFrame(main_frame, text="Nouvelle mesure", padding="10")
        input_frame.grid(row=0, column=0, columnspan=2, sticky=(tk.W, tk.E), pady=5)
        
        # X
        ttk.Label(input_frame, text="Position X (m):").grid(row=0, column=0, sticky=tk.W, padx=5)
        self.x_entry = ttk.Entry(input_frame, width=15)
        self.x_entry.grid(row=0, column=1, padx=5)
        
        # Y
        ttk.Label(input_frame, text="Position Y (m):").grid(row=0, column=2, sticky=tk.W, padx=5)
        self.y_entry = ttk.Entry(input_frame, width=15)
        self.y_entry.grid(row=0, column=3, padx=5)
        
        # Bouton mesure
        self.measure_btn = ttk.Button(
            input_frame, 
            text="📡 Capturer mesure (Enter)", 
            command=self.capture_measurement
        )
        self.measure_btn.grid(row=0, column=4, padx=10)
        
        # Bind Enter key
        self.root.bind('<Return>', lambda e: self.capture_measurement())
        
        # === Section statistiques ===
        stats_frame = ttk.LabelFrame(main_frame, text="Statistiques", padding="10")
        stats_frame.grid(row=1, column=0, columnspan=2, sticky=(tk.W, tk.E), pady=5)
        
        self.stats_label = ttk.Label(stats_frame, text="", font=('Arial', 10))
        self.stats_label.grid(row=0, column=0, sticky=tk.W)
        
        # === Section log ===
        log_frame = ttk.LabelFrame(main_frame, text="Historique des mesures", padding="10")
        log_frame.grid(row=2, column=0, columnspan=2, sticky=(tk.W, tk.E, tk.N, tk.S), pady=5)
        
        self.log_text = scrolledtext.ScrolledText(log_frame, height=15, width=80)
        self.log_text.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        # === Boutons actions ===
        btn_frame = ttk.Frame(main_frame)
        btn_frame.grid(row=3, column=0, columnspan=2, pady=10)
        
        ttk.Button(btn_frame, text="📊 Export CSV", command=self.export_csv).pack(side=tk.LEFT, padx=5)
        ttk.Button(btn_frame, text="🔄 Rafraîchir stats", command=self.update_stats).pack(side=tk.LEFT, padx=5)
        ttk.Button(btn_frame, text="❌ Quitter", command=self.root.quit).pack(side=tk.LEFT, padx=5)
        
        # Configuration du redimensionnement
        self.root.columnconfigure(0, weight=1)
        self.root.rowconfigure(0, weight=1)
        main_frame.columnconfigure(0, weight=1)
        main_frame.rowconfigure(2, weight=1)
        
        # Initialiser les stats
        self.update_stats()
        
    def capture_measurement(self):
        """Capture une mesure"""
        try:
            x = float(self.x_entry.get())
            y = float(self.y_entry.get())
        except ValueError:
            messagebox.showerror("Erreur", "Veuillez entrer des valeurs numériques pour X et Y")
            return
        
        # Désactiver le bouton pendant la mesure
        self.measure_btn.config(state='disabled')
        self.root.update()
        
        try:
            # Ajouter la mesure
            num_measurements, num_networks = self.collector.add_measurement(x, y)
            
            # Log
            msg = f"✅ Mesure capturée à X={x}m, Y={y}m → {num_networks} réseaux détectés\n"
            self.log_text.insert('1.0', msg)
            
            # Réinitialiser les champs
            self.x_entry.delete(0, tk.END)
            self.y_entry.delete(0, tk.END)
            self.x_entry.focus()
            
            # Mettre à jour les stats
            self.update_stats()
            
        except Exception as e:
            messagebox.showerror("Erreur", f"Erreur lors de la capture: {str(e)}")
        finally:
            self.measure_btn.config(state='normal')
    
    def update_stats(self):
        """Met à jour les statistiques"""
        total_bssids, total_measurements = self.collector.get_statistics()
        self.stats_label.config(
            text=f"📡 {total_bssids} points d'accès uniques | "
                 f"📊 {total_measurements} mesures totales"
        )
    
    def export_csv(self):
        """Export en CSV"""
        try:
            self.collector.export_to_csv()
            messagebox.showinfo("Export réussi", "Données exportées dans wifi_data.csv")
            self.log_text.insert('1.0', "💾 Export CSV réussi → wifi_data.csv\n")
        except Exception as e:
            messagebox.showerror("Erreur", f"Erreur lors de l'export: {str(e)}")
    
    def run(self):
        """Lance l'application"""
        self.root.mainloop()


if __name__ == "__main__":
    app = WiFiCollectorGUI()
    app.run()


KeyError: 'measurements'